# Day 14 — SVM Hyperparameter Tuning

Using GridSearchCV and 5-fold cross-validation to find better SVM hyperparameters.

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 2. Load Dataset

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Dataset Shape:', X.shape)
print('\nTarget Classes:', data.target_names)
print('\nClass Distribution:')
print(y.value_counts())

display(X.head())

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training Samples:', X_train.shape[0])
print('Testing Samples :', X_test.shape[0])

## 4. Baseline SVM

Train a standard RBF SVM to create a baseline for comparison.

In [ ]:
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1, gamma='scale'))
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print(f'Baseline SVM Accuracy: {baseline_accuracy:.4f}')

## 5. Define Hyperparameter Grid

- **C:** controls the penalty for classification errors.
- **kernel:** controls the type of decision boundary.
- **gamma:** controls the influence of individual points for RBF/poly kernels.

In [ ]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__gamma': ['scale', 'auto']
}

param_grid

## 6. GridSearchCV with 5-Fold Cross-Validation

In [ ]:
grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print('GridSearchCV completed.')

## 7. Best Hyperparameters

In [ ]:
print('Best Parameters:')
print(grid_search.best_params_)

print('\nBest Cross-Validation Accuracy:')
print(f'{grid_search.best_score_:.4f}')

## 8. Evaluate the Tuned Model

In [ ]:
best_model = grid_search.best_estimator_
tuned_pred = best_model.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_pred)

print(f'Tuned SVM Test Accuracy: {tuned_accuracy:.4f}')

## 9. Baseline vs Tuned SVM

In [ ]:
improvement = tuned_accuracy - baseline_accuracy

comparison = pd.DataFrame({
    'Model': ['Baseline SVM', 'Tuned SVM'],
    'Accuracy': [baseline_accuracy, tuned_accuracy]
})

display(comparison)
print(f'Accuracy Improvement: {improvement:.4f}')

## 10. Classification Report

In [ ]:
print(classification_report(
    y_test,
    tuned_pred,
    target_names=data.target_names
))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, tuned_pred)

print('Confusion Matrix:')
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)
disp.plot()
plt.title('Tuned SVM - Confusion Matrix')
plt.tight_layout()
plt.show()

## 12. Top Hyperparameter Combinations

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)

results = results[[
    'param_svm__C', 'param_svm__kernel', 'param_svm__gamma',
    'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score')

display(results.head(10))

## 13. Visualize Top 10 Combinations

In [ ]:
top_results = results.head(10).copy()

top_results['parameters'] = (
    'C=' + top_results['param_svm__C'].astype(str) +
    ', ' + top_results['param_svm__kernel'].astype(str) +
    ', gamma=' + top_results['param_svm__gamma'].astype(str)
)

plt.figure(figsize=(12, 6))
plt.bar(top_results['parameters'], top_results['mean_test_score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Cross-Validation Accuracy')
plt.xlabel('Hyperparameter Combination')
plt.title('Top SVM Hyperparameter Combinations')
plt.tight_layout()
plt.show()

## 14. Baseline vs Tuned Accuracy

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(comparison['Model'], comparison['Accuracy'])
plt.ylim(min(0.8, comparison['Accuracy'].min() - 0.05), 1.0)
plt.ylabel('Accuracy')
plt.title('Baseline SVM vs Tuned SVM')
plt.tight_layout()
plt.show()

## 15. Final Result

In [ ]:
print('=' * 60)
print('FINAL RESULT')
print('=' * 60)
print(f'Baseline Accuracy : {baseline_accuracy:.4f}')
print(f'Tuned Accuracy    : {tuned_accuracy:.4f}')
print(f'Improvement       : {improvement:.4f}')

print('\nBest Parameters:')
for parameter, value in grid_search.best_params_.items():
    print(f'{parameter}: {value}')

print('\nDay 14 SVM Hyperparameter Tuning Complete!')